# Experiment 6: Named Entity Recognition (NER)

Find names in text and label them: PERSON, ORG, GPE (place), DATE, MONEY, ...

- **IOB tags:** each token is **B**egin, **I**nside or **O**utside an entity.
- **EntityRuler:** add your own rules on top of the model.
- **Evaluation:** Precision = TP/(TP+FP), Recall = TP/(TP+FN), F1 = 2PR/(P+R).

In [ ]:
# !pip install spacy
# !python -m spacy download en_core_web_sm

In [ ]:
import spacy
from spacy import displacy

nlp = spacy.load('en_core_web_sm')
print(len(nlp.get_pipe('ner').labels), 'entity types:', nlp.get_pipe('ner').labels)

In [ ]:
text = ('On 15 March 2026, Priya Sharma, the CEO of Nimbus Technologies, met officials from the '
        'Reserve Bank of India in Mumbai to discuss a $3 billion investment. '
        'Rahul Mehta from Infosys reported a 12% improvement in Bengaluru.')
doc = nlp(text)

for ent in doc.ents:
    print(f'{ent.text:30} {ent.label_:8} {spacy.explain(ent.label_)}')

In [ ]:
displacy.render(doc, style='ent', jupyter=True)   # coloured view

## IOB tags of the first sentence

In [ ]:
for t in list(doc.sents)[0]:
    print(f'{t.text:12} {t.ent_iob_}-{t.ent_type_}' if t.ent_type_ else f'{t.text:12} O')

## Custom rules with EntityRuler

In [ ]:
custom = spacy.load('en_core_web_sm')
ruler = custom.add_pipe('entity_ruler', before='ner')   # rules win over the model
ruler.add_patterns([
    {'label': 'ORG', 'pattern': 'MIT ADT University'},
    {'label': 'TECH', 'pattern': [{'LOWER': 'python'}]},
    {'label': 'TECH', 'pattern': [{'LOWER': 'spacy'}]},
])

s = 'Students at MIT ADT University use Python and spaCy in Pune.'
print('Before:', [(e.text, e.label_) for e in nlp(s).ents])
print('After :', [(e.text, e.label_) for e in custom(s).ents])

## Evaluation on hand-labelled sentences

In [ ]:
gold_data = [
    ('Ananya Iyer joined Google in London in January 2025.',
     {('Ananya Iyer', 'PERSON'), ('Google', 'ORG'), ('London', 'GPE'), ('January 2025', 'DATE')}),
    ('Meera Joshi paid $250 for a flight from Delhi to Paris on 3 May.',
     {('Meera Joshi', 'PERSON'), ('$250', 'MONEY'), ('Delhi', 'GPE'), ('Paris', 'GPE'), ('3 May', 'DATE')}),
    ('Infosys opened a new office in Pune with 2,000 employees.',
     {('Infosys', 'ORG'), ('Pune', 'GPE'), ('2,000', 'CARDINAL')}),
]

tp = fp = fn = 0
for sentence, gold in gold_data:
    pred = {(e.text, e.label_) for e in nlp(sentence).ents}
    tp += len(gold & pred)       # correct
    fp += len(pred - gold)       # predicted but wrong
    fn += len(gold - pred)       # missed

p, r = tp / (tp + fp), tp / (tp + fn)
print(f'Precision={p:.3f}  Recall={r:.3f}  F1={2 * p * r / (p + r):.3f}')